In [1]:
import pandas as pd

files = [
    "data/AlJazeera_articles_Merged.csv",
    "data/bbc_articles_dataset.csv",
    "data/cnn_articles_ALL.csv",
    "data/Dawn_articles_Merged.csv",
    "data/Fox News_articles_Merged.csv",
    "data/trt_articles_Merged_with_content.csv"
]
df_list = [pd.read_csv(f) for f in files]
merged_df = pd.concat(df_list, ignore_index=True)

print("Total rows:", len(merged_df))
merged_df.head()


Total rows: 3553


,Source,Link,Headline,Description,Date,Timestamp,Topic,Author,Region,Article_Content
0,Al Jazeera,https://www.aljazeera.com/opinions/2023/5/17/w...,What is the political agenda of artificial int...,"May 17, 2023 ... We do not believe that it cou...",Last update 17 May 2023,2025-04-23,artificial intelligence,AlJazeera,middle east,Could AI single-handedly decide the course of ...
1,Al Jazeera,https://www.aljazeera.com/news/2017/6/16/faceb...,Facebook reveals AI use to block 'terrorist co...,"Jun 16, 2017 ... Facebook says it has stepped ...",16 Jun 2017,2025-04-23,artificial intelligence,AlJazeera,middle east,US company says technology used to block child...
2,Al Jazeera,https://www.aljazeera.com/features/2017/5/30/c...,Could artificial intelligence lead to world pe...,"May 30, 2017 ... Machines and artificial intel...",30 May 2017,2025-04-23,artificial intelligence,AlJazeera,middle east,Can one man with terminal cancer complete his ...
3,Al Jazeera,https://www.aljazeera.com/opinions/2023/6/9/ar...,Artificial intelligence without borders | US-M...,"Jun 9, 2023 ... The US-Mexico border is alread...",9 Jun 2023,2025-04-23,artificial intelligence,AlJazeera,middle east,The US border-industrial complex has joined th...
4,Al Jazeera,https://www.aljazeera.com/news/2023/5/11/googl...,Google shows the AI evolution of its search en...,"May 11, 2023 ... Google is rolling out more ar...",11 May 2023,2025-04-23,artificial intelligence,AlJazeera,middle east,Google is rolling out more artificial intellig...


In [13]:
merged_df.columns


Index(['Source', 'Headline', 'Description', 'Article_Content'], dtype='object')

In [2]:
use_cols = ["Source", "Headline", "Description", "Article_Content"]
merged_df = merged_df[use_cols]
merged_df = merged_df.rename(
    columns={
        'Source': "source",
        "Headline": "title",
        "Description": "description",
        "Article_Content": "content"
    }
)

# Drop rows with missing content
merged_df = merged_df.dropna(subset=["content"]).reset_index(drop=True)


In [3]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop = set(stopwords.words("english"))

def clean_text(t):
    t = str(t)
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"http\S+", "", t)
    t = t.replace("\n", " ")
    return t.strip()

merged_df["clean_content"] = merged_df["content"].apply(clean_text)


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/shoaib/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [18]:
merged_df.head()

,source,title,description,content,clean_content
0,Al Jazeera,What is the political agenda of artificial int...,"May 17, 2023 ... We do not believe that it cou...",Could AI single-handedly decide the course of ...,Could AI single-handedly decide the course of ...
1,Al Jazeera,Facebook reveals AI use to block 'terrorist co...,"Jun 16, 2017 ... Facebook says it has stepped ...",US company says technology used to block child...,US company says technology used to block child...
2,Al Jazeera,Could artificial intelligence lead to world pe...,"May 30, 2017 ... Machines and artificial intel...",Can one man with terminal cancer complete his ...,Can one man with terminal cancer complete his ...
3,Al Jazeera,Artificial intelligence without borders | US-M...,"Jun 9, 2023 ... The US-Mexico border is alread...",The US border-industrial complex has joined th...,The US border-industrial complex has joined th...
4,Al Jazeera,Google shows the AI evolution of its search en...,"May 11, 2023 ... Google is rolling out more ar...",Google is rolling out more artificial intellig...,Google is rolling out more artificial intellig...


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def get_sentiment(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        output = model(**tokens)
    scores = output.logits.softmax(dim=1).cpu().numpy()
    label = np.argmax(scores) + 1  # 1–5 stars
    return label

merged_df["sentiment"] = merged_df["clean_content"].apply(get_sentiment)
merged_df.head()


/Users/shoaib/Desktop/University/5th Semester/NLP/Project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,source,title,description,content,clean_content,sentiment
0,Al Jazeera,What is the political agenda of artificial int...,"May 17, 2023 ... We do not believe that it cou...",Could AI single-handedly decide the course of ...,Could AI single-handedly decide the course of ...,1
1,Al Jazeera,Facebook reveals AI use to block 'terrorist co...,"Jun 16, 2017 ... Facebook says it has stepped ...",US company says technology used to block child...,US company says technology used to block child...,1
2,Al Jazeera,Could artificial intelligence lead to world pe...,"May 30, 2017 ... Machines and artificial intel...",Can one man with terminal cancer complete his ...,Can one man with terminal cancer complete his ...,4
3,Al Jazeera,Artificial intelligence without borders | US-M...,"Jun 9, 2023 ... The US-Mexico border is alread...",The US border-industrial complex has joined th...,The US border-industrial complex has joined th...,4
4,Al Jazeera,Google shows the AI evolution of its search en...,"May 11, 2023 ... Google is rolling out more ar...",Google is rolling out more artificial intellig...,Google is rolling out more artificial intellig...,5


In [6]:
merged_df.to_csv("data/news_sentiment_final.csv", index=False)